In [2]:
import re
import json
import math
import time
import tempfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report, confusion_matrix
import joblib

In [3]:
PROJECT_DIR = Path(".")
APP_PATH = PROJECT_DIR / "app.exe"
DATA_PATH = PROJECT_DIR / "data" / "urldata.csv"

TARGET_FPR = 0.01
N_REPEATS = 8
RANDOM_STATE = 42

In [4]:
def run_baseline_once(app_path: Path) -> dict:
    proc = subprocess.run(
        [str(app_path)],
        capture_output=True,
        text=True,
        check=True
    )
    out = proc.stdout.strip()

    # Expected output format:
    # Time taken: 225ms
    # FP: 762
    # Total inserts: 344928
    # Negative Queries: 75536
    # Memory needed: 0.394127 MB
    patterns = {
        "time_ms": r"Time taken:\s*(\d+)\s*ms",
        "fp_count": r"FP:\s*(\d+)",
        "insert_count": r"Total inserts:\s*(\d+)",
        "negative_queries": r"Negative Queries:\s*(\d+)",
        "memory_mb": r"Memory needed:\s*([0-9]*\.?[0-9]+)\s*MB",
    }

    result = {}
    for k, p in patterns.items():
        m = re.search(p, out)
        if not m:
            raise ValueError(f"Could not parse {k} from output:\n{out}")
        result[k] = float(m.group(1)) if k == "memory_mb" else int(m.group(1))

    result["observed_fpr"] = result["fp_count"] / max(1, result["negative_queries"])
    return result

In [5]:
baseline_rows = []
for i in range(N_REPEATS):
    row = run_baseline_once(APP_PATH)
    row["trial"] = i + 1
    baseline_rows.append(row)

baseline_df = pd.DataFrame(baseline_rows)

summary = baseline_df.agg({
    "time_ms": ["mean", "std"],
    "memory_mb": ["mean", "std"],
    "observed_fpr": ["mean", "std"],
    "fp_count": ["mean", "std"],
    "insert_count": ["mean", "std"],
    "negative_queries": ["mean", "std"],
})
display(baseline_df)
display(summary)

,time_ms,fp_count,insert_count,negative_queries,memory_mb,observed_fpr,trial
0,170,762,344928,75536,0.394127,0.010088,1
1,172,762,344928,75536,0.394127,0.010088,2
2,159,762,344928,75536,0.394127,0.010088,3
3,168,762,344928,75536,0.394127,0.010088,4
4,177,762,344928,75536,0.394127,0.010088,5
5,169,762,344928,75536,0.394127,0.010088,6
6,155,762,344928,75536,0.394127,0.010088,7
7,166,762,344928,75536,0.394127,0.010088,8


,time_ms,memory_mb,observed_fpr,fp_count,insert_count,negative_queries
mean,167.000000,0.394127,0.010088,762.0,344928.0,75536.0
std,7.050836,0.000000,0.000000,0.0,0.0,0.0


In [6]:
df = pd.read_csv("data/urldata.csv")
df = df[["url", "label"]].dropna().copy()

label_map = {"good": 1, "bad": 0}
df["y"] = df["label"].str.lower().map(label_map)

if df["y"].isna().any():
    bad_values = df.loc[df["y"].isna(), "label"].unique()
    raise ValueError(f"Unexpected label values found: {bad_values}")

print(df.head())
print(df["label"].value_counts())

                      url label  y
0  diaryofagameaddict.com   bad  0
1        espdesign.com.au   bad  0
2      iamagameaddict.com   bad  0
3           kalantzis.net   bad  0
4   slightlyoffcenter.net   bad  0
label
good    344821
bad      75643
Name: count, dtype: int64


In [7]:
# Use raw URL text; HashingVectorizer will build sparse features on the fly.
X = df["url"].astype(str).str.strip()
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (336371,) Test shape: (84093,)


In [8]:
model = Pipeline(steps=[
    ("hash", HashingVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        n_features=2**14,
        alternate_sign=False,
        lowercase=True
    )),
    ("clf", SGDClassifier(
        loss="log_loss",
        alpha=1e-6,
        max_iter=40,
        tol=1e-3,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_score))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

AUC: 0.9928511932120726
Confusion matrix:
 [[14455   674]
 [ 1999 66965]]
              precision    recall  f1-score   support

           0     0.8785    0.9554    0.9154     15129
           1     0.9900    0.9710    0.9804     68964

    accuracy                         0.9682     84093
   macro avg     0.9343    0.9632    0.9479     84093
weighted avg     0.9700    0.9682    0.9687     84093



In [9]:
def model_size_mb(trained_model) -> float:
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        tmp = Path(f.name)
    joblib.dump(trained_model, tmp)
    size_mb = tmp.stat().st_size / (1024 * 1024)
    tmp.unlink(missing_ok=True)
    return size_mb

def run_ml_metrics_once(trained_model, X_eval, y_eval, threshold=0.5):
    # Time prediction only (query phase)
    t0 = time.perf_counter()
    scores = trained_model.predict_proba(X_eval)[:, 1]
    preds = (scores >= threshold).astype(int)
    t1 = time.perf_counter()

    # Confusion matrix with labels [0,1] => [bad,good]
    tn, fp, fn, tp = confusion_matrix(y_eval, preds, labels=[0, 1]).ravel()

    negative_queries = int((y_eval == 0).sum())   # "bad" class as non-member
    observed_fpr = fp / max(1, negative_queries)

    result = {
        "time_taken_ms": (t1 - t0) * 1000.0,
        "fp_count": int(fp),
        "negative_queries": int(negative_queries),
        "observed_fpr": float(observed_fpr),
        "true_positive": int(tp),
        "true_negative": int(tn),
        "false_negative": int(fn),
        "insert_count_analog": int((y_eval == 1).sum()),  # positive class count
        "memory_needed_mb": model_size_mb(trained_model),
    }
    return result

ml_res = run_ml_metrics_once(model, X_test, y_test, threshold=0.5)

In [10]:
# Print in C++-like format
print(f"Time taken: {ml_res['time_taken_ms']:.2f}ms")
print(f"FP: {ml_res['fp_count']}")
print(f"Total inserts: {ml_res['insert_count_analog']}")
print(f"Negative Queries: {ml_res['negative_queries']}")
print(f"Memory needed: {ml_res['memory_needed_mb']:.6f} MB")
print(f"Observed FP rate: {ml_res['observed_fpr']:.6f}")

Time taken: 3514.54ms
FP: 674
Total inserts: 68964
Negative Queries: 15129
Memory needed: 0.126489 MB
Observed FP rate: 0.044550


In [11]:
import hashlib
from dataclasses import dataclass

class SimpleBloomFilter:
    def __init__(self, bit_count: int, hash_count: int):
        self.bit_count = max(8, int(bit_count))
        self.hash_count = max(1, int(hash_count))
        self.bytes = bytearray((self.bit_count + 7) // 8)

    @staticmethod
    def _hash_with_seed(value: str, seed: int) -> int:
        h = hashlib.blake2b(digest_size=8, person=seed.to_bytes(8, "little", signed=False))
        h.update(value.encode("utf-8", errors="ignore"))
        return int.from_bytes(h.digest(), "little", signed=False)

    def _indexes(self, value: str):
        h1 = self._hash_with_seed(value, 0x9E3779B185EBCA87)
        h2 = self._hash_with_seed(value, 0xC2B2AE3D27D4EB4F) or 1
        for i in range(self.hash_count):
            yield (h1 + i * h2) % self.bit_count

    def add(self, value: str):
        for idx in self._indexes(value):
            self.bytes[idx // 8] |= (1 << (idx % 8))

    def contains(self, value: str) -> bool:
        for idx in self._indexes(value):
            if not (self.bytes[idx // 8] & (1 << (idx % 8))):
                return False
        return True

    @property
    def memory_mb(self) -> float:
        return len(self.bytes) / (1024 * 1024)

def bloom_bits_required(n: int, fpr: float) -> int:
    if n <= 0:
        return 8
    ln2 = np.log(2.0)
    m = -(n * np.log(max(1e-12, fpr))) / (ln2 * ln2)
    return int(np.ceil(m))

def bloom_hashes_required(bits: int, n: int) -> int:
    if n <= 0:
        return 1
    return max(1, int(round((bits / n) * np.log(2.0))))

In [12]:
def evaluate_learned_bloom_with_backup(model, df, threshold_grid=None, target_fpr=0.01):
    if threshold_grid is None:
        threshold_grid = np.linspace(0.05, 0.95, 37)

    positive_urls = df.loc[df["y"] == 1, "url"].dropna().astype(str).drop_duplicates().reset_index(drop=True)
    negative_urls = df.loc[df["y"] == 0, "url"].dropna().astype(str).drop_duplicates().reset_index(drop=True)

    pos_scores = model.predict_proba(positive_urls)[:, 1]
    neg_scores = model.predict_proba(negative_urls)[:, 1]

    base_model_mb = model_size_mb(model)
    candidates = []

    for t in threshold_grid:
        model_fp_rate = float((neg_scores >= t).mean())
        if model_fp_rate >= target_fpr:
            continue

        # Keep global FPR at target: p_total = p_model + (1 - p_model) * p_backup
        backup_fpr_target = (target_fpr - model_fp_rate) / max(1e-12, (1.0 - model_fp_rate))
        backup_fpr_target = min(max(backup_fpr_target, 1e-9), 0.99)

        backup_keys = positive_urls[pos_scores < t].tolist()
        bits = bloom_bits_required(len(backup_keys), backup_fpr_target)
        k = bloom_hashes_required(bits, len(backup_keys))
        backup_bf = SimpleBloomFilter(bits, k)
        for key in backup_keys:
            backup_bf.add(key)

        # End-to-end query timing for full system (model + backup checks).
        start = time.perf_counter()
        pos_pred_model = pos_scores >= t
        neg_pred_model = neg_scores >= t

        # Only check backup for items rejected by model to keep this tractable.
        pos_pred_system = pos_pred_model.copy()
        rejected_pos_idx = np.where(~pos_pred_model)[0]
        for idx in rejected_pos_idx:
            if backup_bf.contains(positive_urls.iloc[idx]):
                pos_pred_system[idx] = True

        neg_pred_system = neg_pred_model.copy()
        rejected_neg_idx = np.where(~neg_pred_model)[0]
        for idx in rejected_neg_idx:
            if backup_bf.contains(negative_urls.iloc[idx]):
                neg_pred_system[idx] = True
        end = time.perf_counter()

        fn = int((~pos_pred_system).sum())
        fp = int(neg_pred_system.sum())
        observed_fpr = fp / max(1, len(negative_urls))

        total_mb = base_model_mb + backup_bf.memory_mb
        candidates.append({
            "threshold": float(t),
            "time_taken_ms": (end - start) * 1000.0,
            "insert_count": int(len(positive_urls)),
            "negative_queries": int(len(negative_urls)),
            "false_negative": fn,
            "false_positive": fp,
            "observed_fpr": observed_fpr,
            "model_memory_mb": base_model_mb,
            "backup_memory_mb": backup_bf.memory_mb,
            "total_memory_mb": total_mb,
            "backup_key_count": int(len(backup_keys)),
            "backup_hash_count": int(k),
            "backup_bits": int(bits)
        })

    if not candidates:
        raise RuntimeError("No feasible threshold found. Try a lower model FPR or increase target_fpr.")

    result_df = pd.DataFrame(candidates).sort_values(
        by=["false_negative", "total_memory_mb", "observed_fpr", "time_taken_ms"]
    ).reset_index(drop=True)

    best = result_df.iloc[0].to_dict()
    return best, result_df

In [13]:
best_lbf, lbf_candidates = evaluate_learned_bloom_with_backup(
    model=model,
    df=df,
    target_fpr=TARGET_FPR
)

In [14]:
print("Learned Bloom Filter (model + backup Bloom) metrics")
print(f"Time taken: {best_lbf['time_taken_ms']:.2f}ms")
print(f"FP: {best_lbf['false_positive']}")
print(f"Total inserts: {best_lbf['insert_count']}")
print(f"Negative Queries: {best_lbf['negative_queries']}")
print(f"Memory needed: {best_lbf['total_memory_mb']:.6f} MB")
print(f"Observed FP rate: {best_lbf['observed_fpr']:.6f}")
print(f"False Negatives: {best_lbf['false_negative']}")
print("Memory breakdown (MB):")
print(f"  Model: {best_lbf['model_memory_mb']:.6f}")
print(f"  Backup Bloom: {best_lbf['backup_memory_mb']:.6f}")

display(lbf_candidates.head(10))

Learned Bloom Filter (model + backup Bloom) metrics
Time taken: 561.93ms
FP: 678.0
Total inserts: 344800.0
Negative Queries: 66448.0
Memory needed: 0.169211 MB
Observed FP rate: 0.010203
False Negatives: 0.0
Memory breakdown (MB):
  Model: 0.126489
  Backup Bloom: 0.042723


,threshold,time_taken_ms,insert_count,negative_queries,false_negative,false_positive,observed_fpr,model_memory_mb,backup_memory_mb,total_memory_mb,backup_key_count,backup_hash_count,backup_bits
0,0.800,561.9291,344800,66448,0,678,0.010203,0.126489,0.042723,0.169211,27418,9,358377
1,0.775,602.2956,344800,66448,0,665,0.010008,0.126489,0.042753,0.169242,24341,10,358638
2,0.825,538.0757,344800,66448,0,656,0.009872,0.126489,0.044723,0.171211,30907,8,375159
3,0.850,649.8033,344800,66448,0,676,0.010173,0.126489,0.048776,0.175264,35289,8,409156
4,0.875,628.3208,344800,66448,0,682,0.010264,0.126489,0.054496,0.180984,40971,8,457139
5,0.900,637.9832,344800,66448,0,679,0.010219,0.126489,0.061874,0.188363,48269,7,519039
6,0.925,699.9685,344800,66448,0,667,0.010038,0.126489,0.072919,0.199408,58731,7,611688
7,0.950,819.4463,344800,66448,0,642,0.009662,0.126489,0.090807,0.217296,75294,7,761741
